This notebook contains Python code for reproducing the results in our paper on comparing dynamic LLM-generated feedback to static feedback for automatically generated fill-in-the-blank practice questions delivered in a textbook ereader platform:

Johnson, B. G., Dittel, J. S., Ortiz, O. J., Bistolfi, R., Clark, M. W., Jerome, B., Benton, R., & Van Campenhout, R. (2026). LLM feedback isn't automatically better: Static scaffolds outperform dynamic feedback in textbook-embedded practice. In S. Sosnovsky, P. Brusilovsky, A. Lan, & I. Alpizar-Chacon (Eds.), *Proceedings of the Seventh International Workshop on Intelligent Textbooks 2026* (pp. 4–18). CEUR Workshop Proceedings, Vol. 4231. [https://ceur-ws.org/Vol-4231/itb26_s1p1.pdf](https://ceur-ws.org/Vol-4231/itb26_s1p1.pdf)

This paper was presented at [AIED 2026](https://www.aied-conference.org/2026) as part of the [Seventh Workshop on Intelligent Textbooks (iTextbooks)](https://intextbooks.science.uu.nl/workshop2026/).

Results are presented in the order they occur, organized by the paper's sections. For each result, an excerpt from the paper is given followed by code to compute the result from the dataset provided. Example:

>The final dataset contains 33,834 sessions across 23,148 questions, 5,596 students, and 1,363 textbooks.

```len( sessions ), sessions.question_id.nunique(), sessions.student_id.nunique(), sessions.textbook_id.nunique()```

Please refer to the paper for additional context.

In [1]:
import numpy as np
import pandas as pd

In [2]:
%load_ext rpy2.ipython

In [3]:
%%R
library( arrow )
library( glmmTMB )

Some features are not enabled in this build of Arrow. Run `arrow_info()` for more information.
The repository you retrieved Arrow from did not include all of Arrow's features.
You can install a fully-featured version by running:
`install.packages('arrow', repos = 'https://apache.r-universe.dev')`.

Attaching package: ‘arrow’

The following object is masked from ‘package:utils’:

    timestamp



## Read dataset of student-question sessions

In [4]:
sessions = pd.read_parquet( 'sessions.parquet' )
sessions.head()

,timestamp,student_id,question_id,textbook_id,subject,question_stem,correct_answer,student_answer,assigned_condition,realized_condition,feedback_text,next_action,answer_reveal,correct_retry,edit_distance,edit_distance_quartile
0,2026-04-09 16:47:34,RN4R7PMVJZ3HP4QCY7VQ,1c1a04325be5e658547de661371e44373920d4c7bab976...,9781071925287,Business & Economics,"In conjunction with the latter view, it has be...",global,set,dynamic,common_fallback,The same answer also completes the following s...,incorrect_retry,0,0,6,Q1: 1–6
1,2026-04-09 16:48:40,RN4R7PMVJZ3HP4QCY7VQ,c7f7b15894ef8d1945dfef087d7055a6aa9a02e09937f8...,9781071925287,Business & Economics,With each successful ______ business venture (...,international,set,static,context_assigned,Here's a bit more to help you. Try again. <spa...,answer_reveal,1,0,11,Q4: 11+
2,2026-04-09 16:49:31,RN4R7PMVJZ3HP4QCY7VQ,6dc15f0bd8663f347219248a28a5e81ceea837f35b5319...,9781071925287,Business & Economics,People from more collectivist cultures may be ...,constructive,set,dynamic,dynamic,"You said ""set,"" but that doesn't fit with the ...",answer_reveal,1,0,10,Q3: 9–10
3,2026-04-09 16:49:57,SNVHNNPFVD2F7CT8ES3W,880ba64e728318c6645177a3d5a98731acaa111d453597...,9781544309217,Social Science,A textbook ______ gives the impression—incorre...,definition,on drugs,dynamic,dynamic,"You said ""on drugs,"" but that doesn't relate t...",no_action,0,0,10,Q3: 9–10
4,2026-04-09 16:50:20,PWMWHRFF22TCDTTE25UK,0a998b1f6d8bc9e2c6ae090fd85a5cd2c5f71b715eec08...,9781317342465,Social Science,"As such, women's religions center on acknowled...",healing,embodiment,dynamic,dynamic,"You said ""embodiment,"" but this word refers to...",incorrect_retry,0,0,8,Q2: 7–8


## 2. Method

### 2.1. Static and dynamic feedback generation

>The dynamic condition uses an LLM to generate feedback based on the student’s actual incorrect answer. The model used in this deployment was GPT-4.1 nano [32]. For each incorrect attempt, the model is given three inputs: the question stem, the correct answer, and the student’s incorrect answer. The feedback generation prompt instructs the model to produce brief supportive feedback that acknowledges the student’s answer, explains why it does not fit, redirects the student without revealing the correct answer, and encourages a retry. ... The generated response is constrained to two or three sentences. The full prompt, along with the dataset used in this study, will be made available in our open data repository [33].

In [5]:
with open( 'dynamic_feedback_prompt.txt', 'r' ) as f:
    print( f.read() )

Dynamic Feedback Prompt — LLM Feedback Isn't Automatically Better (iTextbooks 2026)

Model: gpt-4.1-nano
Temperature: 0.0

This file documents the prompt used to generate dynamic, error-sensitive feedback for
incorrect fill-in-the-blank (FITB) responses in the VitalSource CoachMe platform. The
prompt is split into a system message and a user message, as sent to the OpenAI Chat
Completions API. Placeholders shown in curly braces — {question}, {correct_answer},
{student_answer} — are substituted with the actual values for each session at runtime.

--------------------------------------------------------------------------------------
SYSTEM MESSAGE
--------------------------------------------------------------------------------------

You are an AI tutor giving feedback on student answers to practice questions. You are
very professional. All your responses are ethical. You never use profanity and you never
respond to messages that violate any ethical, moral or legal standard.

-----------

### 2.3. Observation window, outcomes, and analysis plan

>The final dataset contains 33,834 sessions across 23,148 questions, 5,596 students, and 1,363 textbooks.

In [6]:
len( sessions ), sessions.question_id.nunique(), sessions.student_id.nunique(), sessions.textbook_id.nunique()

(33834, 23148, 5596, 1363)

>Using the Book Industry Standards and Communications major subject heading classification [36] available for most of the textbooks, the top subject domains as a percentage of session data were Psychology (22.7%), Social Science (19.1%), Political Science (15.7%), Business & Economics (12.2%), and Law (7.5%).

In [7]:
sessions.subject.value_counts( normalize=True ).apply( lambda p: f'{p:.1%}' )

subject
Psychology                     22.7%
Social Science                 19.1%
Political Science              15.7%
Business & Economics           12.2%
Law                             7.5%
Education                       4.7%
Language Arts & Disciplines     4.4%
Medical                         3.3%
MISSING                         1.3%
Science                         1.1%
History                         1.1%
Art                             1.1%
Health & Fitness                1.0%
Family & Relationships          0.8%
Computers                       0.8%
Technology & Engineering        0.7%
Sports & Recreation             0.7%
Study Aids                      0.4%
Nature                          0.4%
Religion                        0.4%
Philosophy                      0.2%
Transportation                  0.1%
Juvenile Nonfiction             0.1%
Music                           0.0%
Performing Arts                 0.0%
Architecture                    0.0%
Juvenile Fiction              

## 3. Results and Discussion

### 3.1. Randomized comparison

>**Table 1**<br/>Randomized ITT and CACE estimates for answer reveal and correct retry.

In [8]:
analysis = sessions[ [ 'student_id', 'assigned_condition', 'realized_condition',
                        'answer_reveal', 'correct_retry' ] ].copy()
analysis[ 'Z' ] = ( analysis.assigned_condition == 'dynamic' ).astype( int )
analysis[ 'D' ] = ( analysis.realized_condition == 'dynamic' ).astype( int )
analysis[ 'Y_answer_reveal' ] = analysis.answer_reveal.astype( float )
analysis[ 'Y_correct_retry' ] = analysis.correct_retry.astype( float )

In [9]:
def cluster_bootstrap_itt_cace(
    data, outcome_col, cluster_col='student_id',
    n_boot=100_000, random_state=42, progress_every=10_000,
):
    rng = np.random.default_rng( random_state )
    cluster_codes, cluster_labels = pd.factorize( data[ cluster_col ], sort=False )
    n_clusters = len( cluster_labels )
    print( n_clusters, 'clusters' )

    Z_dyn = ( data.assigned_condition == 'dynamic' ).to_numpy()
    Z_sta = ( data.assigned_condition == 'static' ).to_numpy()
    D     = data.D.to_numpy( dtype=float )
    Y     = pd.to_numeric( data[ outcome_col ], errors='coerce' ).to_numpy( dtype=float )

    rows = np.empty( ( n_boot, 5 ), dtype=float )
    for b in range( n_boot ):
        if ( b + 1 ) % progress_every == 0 or b + 1 == n_boot:
            print( f'Sample {b+1} of {n_boot}' )
        sampled        = rng.integers( 0, n_clusters, size=n_clusters )
        cluster_counts = np.bincount( sampled, minlength=n_clusters )
        w              = cluster_counts[ cluster_codes ].astype( float )
        w_dyn, w_sta   = w[ Z_dyn ], w[ Z_sta ]
        mean_dyn       = np.sum( Y[ Z_dyn ] * w_dyn ) / np.sum( w_dyn )
        mean_sta       = np.sum( Y[ Z_sta ] * w_sta ) / np.sum( w_sta )
        itt            = mean_dyn - mean_sta
        compliance     = np.sum( D[ Z_dyn ] * w_dyn ) / np.sum( w_dyn )
        cace           = itt / compliance if compliance != 0 else np.nan
        rows[ b ]      = ( mean_dyn, mean_sta, itt, compliance, cace )

    draws   = pd.DataFrame( rows, columns=[ 'mean_dynamic', 'mean_static',
                                            'ITT_difference', 'compliance_rate', 'CACE' ] )
    summary = draws.agg( [ 'mean', 'median' ] ).T.rename(
                         columns={ 'mean': 'boot_mean', 'median': 'boot_median' } )
    summary[ 'ci_2.5' ]  = draws.quantile( 0.025 )
    summary[ 'ci_97.5' ] = draws.quantile( 0.975 )
    return draws, summary

In [10]:
boot_reveal_draws, boot_reveal_summary = cluster_bootstrap_itt_cace(
    analysis, 'Y_answer_reveal', n_boot=100_000, progress_every=10_000 )
boot_retry_draws, boot_retry_summary   = cluster_bootstrap_itt_cace(
    analysis, 'Y_correct_retry', n_boot=100_000, progress_every=10_000 )

5596 clusters
Sample 10000 of 100000
Sample 20000 of 100000
Sample 30000 of 100000
Sample 40000 of 100000
Sample 50000 of 100000
Sample 60000 of 100000
Sample 70000 of 100000
Sample 80000 of 100000
Sample 90000 of 100000
Sample 100000 of 100000
5596 clusters
Sample 10000 of 100000
Sample 20000 of 100000
Sample 30000 of 100000
Sample 40000 of 100000
Sample 50000 of 100000
Sample 60000 of 100000
Sample 70000 of 100000
Sample 80000 of 100000
Sample 90000 of 100000
Sample 100000 of 100000


In [11]:
def fmt( point, ci_lo, ci_hi, scale=100 ):
    return f'{point*scale:.1f} [{ci_lo*scale:.1f}, {ci_hi*scale:.1f}]'

In [12]:
rates = analysis.groupby( 'assigned_condition' )[ [ 'Y_answer_reveal', 'Y_correct_retry' ] ].mean()

dyn_reveal = rates.loc[ 'dynamic', 'Y_answer_reveal' ]
sta_reveal = rates.loc[ 'static',  'Y_answer_reveal' ]
dyn_retry  = rates.loc[ 'dynamic', 'Y_correct_retry' ]
sta_retry  = rates.loc[ 'static',  'Y_correct_retry' ]

itt_reveal = dyn_reveal - sta_reveal
itt_retry  = dyn_retry  - sta_retry

compliance  = analysis.loc[ analysis.assigned_condition == 'dynamic', 'D' ].mean()
cace_reveal = itt_reveal / compliance
cace_retry  = itt_retry  / compliance

table_1 = pd.DataFrame( {
    'Outcome':       [ 'Answer reveal', 'Correct retry' ],
    'Dynamic (%)':   [ round( dyn_reveal * 100, 1 ), round( dyn_retry * 100, 1 ) ],
    'Static (%)':    [ round( sta_reveal * 100, 1 ), round( sta_retry * 100, 1 ) ],
    'ITT [95% CI]':  [ fmt( itt_reveal,
                            boot_reveal_summary.loc[ 'ITT_difference', 'ci_2.5' ],
                            boot_reveal_summary.loc[ 'ITT_difference', 'ci_97.5' ] ),
                       fmt( itt_retry,
                            boot_retry_summary.loc[ 'ITT_difference', 'ci_2.5' ],
                            boot_retry_summary.loc[ 'ITT_difference', 'ci_97.5' ] ) ],
    'CACE [95% CI]': [ fmt( cace_reveal,
                            boot_reveal_summary.loc[ 'CACE', 'ci_2.5' ],
                            boot_reveal_summary.loc[ 'CACE', 'ci_97.5' ] ),
                       fmt( cace_retry,
                            boot_retry_summary.loc[ 'CACE', 'ci_2.5' ],
                            boot_retry_summary.loc[ 'CACE', 'ci_97.5' ] ) ],
} ).set_index( 'Outcome' )
table_1

,Dynamic (%),Static (%),ITT [95% CI],CACE [95% CI]
Outcome,,,,
Answer reveal,61.0,64.3,"-3.3 [-4.4, -2.2]","-5.3 [-7.2, -3.6]"
Correct retry,16.5,16.9,"-0.4 [-1.3, 0.4]","-0.7 [-2.1, 0.7]"


>The reduction in answer reveal did not translate into more students recovering the correct answer on the next action. Incorrect retry rates were correspondingly higher under dynamic (18.5%) than static (14.6%), consistent with the persistence shift being absorbed primarily by failed rather than successful retries.

In [13]:
( sessions.next_action == 'incorrect_retry' ).groupby( sessions.assigned_condition ).mean().apply( lambda p: f'{p:.1%}' )

assigned_condition
dynamic    18.5%
static     14.6%
Name: next_action, dtype: object

>Compliance with dynamic delivery was moderate rather than high (61.6%), reflecting fallback under the no-leak guardrail in a substantial minority of dynamic-assigned cases.

In [14]:
print( f'{compliance:.1%}' )

61.6%


>**Table 2**<br/>Mixed effects logistic regression results for the randomized ITT comparison. The assigned dynamic condition is the reference level.

In [15]:
%%R
sessions <- read_parquet( 'sessions.parquet' )

#### Answer reveal

In [16]:
%%R
sessions$assigned_condition <- relevel( factor( sessions$assigned_condition ), ref = 'dynamic' )
model <- glmmTMB( answer_reveal ~ assigned_condition + (1|student_id) + (1|question_id),
                  family=binomial(link=logit), data=sessions )

In [17]:
%%R
summary( model )

 Family: binomial  ( logit )
Formula:          answer_reveal ~ assigned_condition + (1 | student_id) + (1 |  
    question_id)
Data: sessions

     AIC      BIC   logLik deviance df.resid 
 31093.9  31127.7 -15543.0  31085.9    33830 

Random effects:

Conditional model:
 Groups      Name        Variance Std.Dev.
 student_id  (Intercept) 5.22669  2.2862  
 question_id (Intercept) 0.05071  0.2252  
Number of obs: 33834, groups:  student_id, 5596; question_id, 23148

Conditional model:
                         Estimate Std. Error z value Pr(>|z|)    
(Intercept)              -0.07543    0.04460  -1.691   0.0908 .  
assigned_conditionstatic  0.30515    0.03473   8.788   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [18]:
%%R
round( exp( cbind( OR=fixef( model )$cond ) ), 2 )

                           OR
(Intercept)              0.93
assigned_conditionstatic 1.36


#### Correct retry

In [19]:
%%R
model <- glmmTMB( correct_retry ~ assigned_condition + (1|student_id) + (1|question_id),
                  family=binomial(link=logit), data=sessions )

In [20]:
%%R
summary( model )

 Family: binomial  ( logit )
Formula:          correct_retry ~ assigned_condition + (1 | student_id) + (1 |  
    question_id)
Data: sessions

     AIC      BIC   logLik deviance df.resid 
 23565.8  23599.5 -11778.9  23557.8    33830 

Random effects:

Conditional model:
 Groups      Name        Variance Std.Dev.
 student_id  (Intercept) 5.8417   2.4170  
 question_id (Intercept) 0.3097   0.5565  
Number of obs: 33834, groups:  student_id, 5596; question_id, 23148

Conditional model:
                         Estimate Std. Error z value Pr(>|z|)    
(Intercept)              -2.66801    0.07987  -33.41   <2e-16 ***
assigned_conditionstatic  0.07274    0.04134    1.76   0.0785 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [21]:
%%R
round( exp( cbind( OR=fixef( model )$cond ) ), 2 )

                           OR
(Intercept)              0.07
assigned_conditionstatic 1.08


### 3.2. Realized condition decomposition

>**Table 3**<br/>Descriptive outcome rates by realized feedback condition.

In [22]:
sessions.groupby( 'realized_condition' )[ [ 'answer_reveal', 'correct_retry' ] ].mean().mul( 100 ).round( 1 )

,answer_reveal,correct_retry
realized_condition,,
common_assigned,63.1,18.4
common_fallback,55.5,21.3
context_assigned,63.2,16.2
context_fallback,57.7,17.8
dynamic,63.6,14.5
outcome_assigned,70.4,12.9
outcome_fallback,60.9,16.9


>Cell sizes differed across realized feedback conditions, ranging from 1,098 to 10,410.

In [23]:
sessions.realized_condition.value_counts()

realized_condition
dynamic             10410
common_assigned      9557
context_assigned     4580
common_fallback      3623
outcome_assigned     2789
context_fallback     1777
outcome_fallback     1098
Name: count, dtype: int64

>Common answer feedback accounted for 56.3% of static-delivered cases.

In [24]:
static_delivered = sessions[ sessions.realized_condition != 'dynamic' ]
is_common = static_delivered.realized_condition.str.startswith( 'common' )
print( f'{is_common.mean():.1%}' )

56.3%


>**Table 4**<br/>Mixed effects logistic regression results for the realized feedback condition decomposition. Dynamic feedback is the reference level.

#### Panel A. Answer reveal

In [25]:
%%R
sessions$realized_condition <- relevel( factor( sessions$realized_condition ), ref = 'dynamic' )
model <- glmmTMB( answer_reveal ~ realized_condition + (1|student_id) + (1|question_id),
                  family=binomial(link=logit), data=sessions )

In [26]:
%%R
summary( model )

 Family: binomial  ( logit )
Formula:          answer_reveal ~ realized_condition + (1 | student_id) + (1 |  
    question_id)
Data: sessions

     AIC      BIC   logLik deviance df.resid 
 30980.7  31056.6 -15481.3  30962.7    33825 

Random effects:

Conditional model:
 Groups      Name        Variance Std.Dev.
 student_id  (Intercept) 5.27632  2.297   
 question_id (Intercept) 0.05617  0.237   
Number of obs: 33834, groups:  student_id, 5596; question_id, 23148

Conditional model:
                                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)                        -0.09270    0.04925  -1.882 0.059808 .  
realized_conditioncommon_assigned   0.16500    0.04540   3.634 0.000279 ***
realized_conditioncommon_fallback  -0.11379    0.05992  -1.899 0.057557 .  
realized_conditioncontext_assigned  0.30779    0.05736   5.366 8.05e-08 ***
realized_conditioncontext_fallback  0.09189    0.08082   1.137 0.255543    
realized_conditionoutcome_assigned  0.88446    0.07250 

In [27]:
%%R
round( exp( cbind( OR=fixef( model )$cond ) ), 2 )

                                     OR
(Intercept)                        0.91
realized_conditioncommon_assigned  1.18
realized_conditioncommon_fallback  0.89
realized_conditioncontext_assigned 1.36
realized_conditioncontext_fallback 1.10
realized_conditionoutcome_assigned 2.42
realized_conditionoutcome_fallback 1.50


#### Panel B. Correct retry

In [28]:
%%R
model <- glmmTMB( correct_retry ~ realized_condition + (1|student_id) + (1|question_id),
                  family=binomial(link=logit), data=sessions )

In [29]:
%%R
summary( model )

 Family: binomial  ( logit )
Formula:          correct_retry ~ realized_condition + (1 | student_id) + (1 |  
    question_id)
Data: sessions

     AIC      BIC   logLik deviance df.resid 
 23468.1  23544.0 -11725.1  23450.1    33825 

Random effects:

Conditional model:
 Groups      Name        Variance Std.Dev.
 student_id  (Intercept) 5.9127   2.4316  
 question_id (Intercept) 0.3121   0.5587  
Number of obs: 33834, groups:  student_id, 5596; question_id, 23148

Conditional model:
                                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)                        -2.72708    0.08408  -32.43  < 2e-16 ***
realized_conditioncommon_assigned   0.33755    0.05480    6.16 7.27e-10 ***
realized_conditioncommon_fallback   0.31069    0.07079    4.39 1.14e-05 ***
realized_conditioncontext_assigned -0.04613    0.06974   -0.66    0.508    
realized_conditioncontext_fallback -0.11231    0.09752   -1.15    0.249    
realized_conditionoutcome_assigned -0.38538    0.08781 

In [30]:
%%R
round( exp( cbind( OR=fixef( model )$cond ) ), 2 )

                                     OR
(Intercept)                        0.07
realized_conditioncommon_assigned  1.40
realized_conditioncommon_fallback  1.36
realized_conditioncontext_assigned 0.95
realized_conditioncontext_fallback 0.89
realized_conditionoutcome_assigned 0.68
realized_conditionoutcome_fallback 0.84


>Using edit distance between the student’s answer and the correct answer as a proxy for response proximity, we found that fallback was more likely for responses closer to the correct answer, suggesting that leakage risk is greater when the student answer is already close to the target term.

In [31]:
dynamic_assigned = sessions[ sessions.assigned_condition == 'dynamic' ]
fallback = dynamic_assigned.realized_condition != 'dynamic'
fallback.groupby( dynamic_assigned.edit_distance_quartile ).mean().apply( lambda p: f'{p:.1%}' )

edit_distance_quartile
Q1: 1–6     46.2%
Q2: 7–8     38.8%
Q3: 9–10    35.0%
Q4: 11+     29.7%
Name: realized_condition, dtype: object

Fallback rates decrease monotonically across quartiles, from 46.2% for responses closest to the correct answer (Q1) to 29.7% for those furthest away (Q4), consistent with the paper's finding that leakage risk is greater when the student answer is already close to the target term.

>However, descriptive differences between conditions remained similar across edit distance strata, and adding
edit distance quartile as a covariate did not materially change the assigned condition effect in the mixed effects models.

#### Answer reveal

In [32]:
%%R
model <- glmmTMB( answer_reveal ~ assigned_condition + edit_distance_quartile + (1|student_id) + (1|question_id),
                  family=binomial(link=logit), data=sessions )

In [33]:
%%R
summary( model )

 Family: binomial  ( logit )
Formula:          
answer_reveal ~ assigned_condition + edit_distance_quartile +  
    (1 | student_id) + (1 | question_id)
Data: sessions

     AIC      BIC   logLik deviance df.resid 
 31077.1  31136.1 -15531.5  31063.1    33827 

Random effects:

Conditional model:
 Groups      Name        Variance Std.Dev.
 student_id  (Intercept) 5.20483  2.2814  
 question_id (Intercept) 0.05029  0.2242  
Number of obs: 33834, groups:  student_id, 5596; question_id, 23148

Conditional model:
                               Estimate Std. Error z value Pr(>|z|)    
(Intercept)                    -0.21095    0.05341  -3.950 7.81e-05 ***
assigned_conditionstatic        0.30859    0.03474   8.882  < 2e-16 ***
edit_distance_quartileQ2: 7–8   0.14940    0.04497   3.322 0.000893 ***
edit_distance_quartileQ3: 9–10  0.20999    0.04805   4.370 1.24e-05 ***
edit_distance_quartileQ4: 11+   0.20350    0.05915   3.440 0.000581 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 

The `assigned_condition` coefficient remains statistically significant and similar in magnitude to the primary ITT model (Table 2), confirming that the answer reveal result is not driven by response proximity to the correct answer.

#### Correct retry

In [34]:
%%R
model <- glmmTMB( correct_retry ~ assigned_condition + edit_distance_quartile + (1|student_id) + (1|question_id),
                  family=binomial(link=logit), data=sessions )

In [35]:
%%R
summary( model )

 Family: binomial  ( logit )
Formula:          
correct_retry ~ assigned_condition + edit_distance_quartile +  
    (1 | student_id) + (1 | question_id)
Data: sessions

     AIC      BIC   logLik deviance df.resid 
 23504.3  23563.3 -11745.1  23490.3    33827 

Random effects:

Conditional model:
 Groups      Name        Variance Std.Dev.
 student_id  (Intercept) 5.7213   2.392   
 question_id (Intercept) 0.3102   0.557   
Number of obs: 33834, groups:  student_id, 5596; question_id, 23148

Conditional model:
                               Estimate Std. Error z value Pr(>|z|)    
(Intercept)                    -2.38898    0.08456 -28.251  < 2e-16 ***
assigned_conditionstatic        0.06811    0.04137   1.647   0.0997 .  
edit_distance_quartileQ2: 7–8  -0.28022    0.05310  -5.277 1.31e-07 ***
edit_distance_quartileQ3: 9–10 -0.37219    0.05747  -6.476 9.40e-11 ***
edit_distance_quartileQ4: 11+  -0.51126    0.07305  -6.999 2.59e-12 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 

As in the primary ITT model (Table 2), `assigned_condition` does not reach statistical significance for correct retry. The non-significant result here is consistent with the paper's conclusion: controlling for edit distance does not change the qualitative finding that dynamic and static feedback produce no detectable difference in correct retry.